# Exploração dos primos gerados por $P(b, n) = (b^n) \bmod ((b-1)^n)$

Este notebook permite investigar valores de `b` (base) e `n` (expoente) tais que
\[ P(b, n) = (b^n) \bmod ((b-1)^n) \]
seja um número primo. Para utilizá-lo no Google Colab:

⚠️ **Não execute comandos que começam com `(cd "$(git ...` ou `git apply`.** Esses trechos aparecem em algumas instruções automatizadas, mas não fazem parte do notebook. Execute apenas as células de Python abaixo.

1. Execute a célula **Funções utilitárias** para carregar os métodos de cálculo e teste de primalidade.
2. Utilize uma das células interativas (identificadas com o título na parte superior) para verificar um par específico `(b, n)`, listar resultados para uma base fixa ou buscar várias ocorrências em intervalos.
3. Ajuste os campos dos formulários conforme necessário e execute cada célula individualmente.

> **Importante:** todas as células executam apenas código Python. Não há necessidade de rodar comandos de terminal como `git apply`.



In [ ]:
# Funções utilitárias
from typing import Iterable, List, Tuple

def miller_rabin(n: int) -> bool:
    """Teste determinístico de primalidade para inteiros de 64 bits.

    Para números maiores o teste segue altamente confiável por utilizar uma combinação
    de bases conhecidas.
    """
    if n < 2:
        return False
    small_primes = (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37)
    if n in small_primes:
        return True
    if any(n % p == 0 for p in small_primes):
        return False

    # escreve n - 1 como d * 2^s
    d = n - 1
    s = 0
    while d % 2 == 0:
        d //= 2
        s += 1

    def try_composite(a: int) -> bool:
        x = pow(a, d, n)
        if x in (1, n - 1):
            return False
        for _ in range(s - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                return False
        return True

    # Bases suficientes para todos os inteiros de 64 bits
    test_bases = (2, 3, 5, 7, 11, 13, 17)

    return not any(try_composite(a) for a in test_bases if a < n)


def calcular_P(b: int, n: int) -> int:
    """Calcula P(b, n) = (b**n) mod ((b-1)**n)."""
    if b <= 1:
        raise ValueError("A base b deve ser maior que 1.")
    if n < 1:
        raise ValueError("O expoente n deve ser positivo.")
    modulo = pow(b - 1, n)
    return pow(b, n, modulo)


def gerar_primos(limit_b: int, limit_n: int) -> List[Tuple[int, int, int]]:
    """Gera todos os pares (b, n) até os limites dados em que P(b, n) é primo."""
    resultados: List[Tuple[int, int, int]] = []
    for b in range(2, limit_b + 1):
        for n in range(1, limit_n + 1):
            valor = calcular_P(b, n)
            if valor > 1 and miller_rabin(valor):
                resultados.append((b, n, valor))
    return resultados


def primeiros_primos(limit_b: int, limit_n: int, quantidade: int) -> List[Tuple[int, int, int]]:
    """Retorna as primeiras ocorrências onde P(b, n) é primo.

    A busca é feita crescendo b e, para cada b, crescendo n.
    """
    encontrados: List[Tuple[int, int, int]] = []
    for b in range(2, limit_b + 1):
        for n in range(1, limit_n + 1):
            valor = calcular_P(b, n)
            if valor > 1 and miller_rabin(valor):
                encontrados.append((b, n, valor))
                if len(encontrados) >= quantidade:
                    return encontrados
    return encontrados


def listar_primos_da_base(b: int, limite_n: int) -> Iterable[Tuple[int, int, int]]:
    """Gera resultados primos apenas para uma base fixa."""
    for n in range(1, limite_n + 1):
        valor = calcular_P(b, n)
        if valor > 1 and miller_rabin(valor):
            yield b, n, valor



In [ ]:
#@title Verificar se $P(b, n)$ é primo
try:
    calcular_P
except NameError:
    raise RuntimeError('Execute a célula \"Funções utilitárias\" antes de usar este formulário.')

b = 4  #@param {type:"integer"}
n = 2  #@param {type:"integer"}
valor = calcular_P(b, n)
print(f"P({b}, {n}) = {valor}")
print("É primo?", "Sim" if miller_rabin(valor) else "Não")



In [ ]:
#@title Listar primos para uma base fixa
try:
    listar_primos_da_base
except NameError:
    raise RuntimeError('Execute a célula \"Funções utilitárias\" antes de usar este formulário.')

base = 5  #@param {type:"integer"}
limite_n = 25  #@param {type:"integer"}
resultados = list(listar_primos_da_base(base, limite_n))

if resultados:
    for _, n, valor in resultados:
        print(f"n={n} => P({base}, {n}) = {valor}")
else:
    print("Nenhum valor primo encontrado para a base escolhida no intervalo informado.")



In [ ]:
#@title Buscar valores primos em intervalos
try:
    primeiros_primos
except NameError:
    raise RuntimeError('Execute a célula \"Funções utilitárias\" antes de usar este formulário.')

limite_b = 15  #@param {type:"integer"}
limite_n = 10  #@param {type:"integer"}
quantidade = 10  #@param {type:"integer"}
primos_encontrados = primeiros_primos(limite_b, limite_n, quantidade)

if primos_encontrados:
    for b, n, valor in primos_encontrados:
        print(f"b={b}, n={n} => P(b, n) = {valor}")
    if len(primos_encontrados) < quantidade:
        print(f"\nForam encontrados apenas {len(primos_encontrados)} resultados dentro dos limites informados.")
else:
    print("Nenhum P(b, n) primo encontrado nos intervalos informados.")

